# Privacy & Data Minimization

This notebook uses fictional customer records to demonstrate data minimization, masking, and pseudonymization.

**Learning goals**
- Keep only fields needed for a stated task.
- Recognize that masking and pseudonyms are not the same as anonymization.
- See why a re-identification key must be protected.

No real personal data is used.

In [ ]:
records = [
    {"name": "Avery Example", "email": "avery@example.invalid", "age": 34, "city": "Harborview", "plan": "basic", "support_issue": "password reset"},
    {"name": "Blake Sample", "email": "blake@example.invalid", "age": 42, "city": "Pine Ridge", "plan": "pro", "support_issue": "billing question"},
    {"name": "Casey Demo", "email": "casey@example.invalid", "age": 29, "city": "Harborview", "plan": "basic", "support_issue": "password reset"},
]
records

## 1. Minimize for the task

Suppose the task is to count issue types. Names, email addresses, exact ages, and cities are not needed. Removing unused fields reduces exposure.

In [ ]:
from collections import Counter

issue_only = [{"support_issue": row["support_issue"]} for row in records]
issue_counts = Counter(row["support_issue"] for row in issue_only)
print("Minimized rows:", issue_only)
print("Counts:", dict(issue_counts))

## 2. Masking and pseudonymization

Masking changes how a value is displayed. Pseudonymization replaces identifiers with aliases while retaining a separate lookup key. Neither guarantees anonymity: combinations of remaining attributes can still identify someone.

In [ ]:
masked_email = records[0]["email"].replace("avery", "a***")
print("Masked for display:", masked_email)

aliases = {row["email"]: f"PERSON-{i+1:03d}" for i, row in enumerate(records)}
lookup_key = {alias: email for email, alias in aliases.items()}  # protect separately
pseudonymized = [
    {"person_id": aliases[row["email"]], "plan": row["plan"], "support_issue": row["support_issue"]}
    for row in records
]
print("Pseudonymized dataset:", pseudonymized)
print("The lookup key can reconnect aliases to email addresses; restrict and protect it.")

## 3. Coarsen and suppress small groups

Aggregates can still reveal people when groups are very small. This toy policy suppresses categories with fewer than two records.

In [ ]:
threshold = 2
published = {
    issue: count if count >= threshold else "SUPPRESSED"
    for issue, count in issue_counts.items()
}
print("Thresholded aggregate:", published)

## Takeaways

- Start from the purpose and collect only the fields needed for it.
- Masking is display protection; pseudonymization is reversible when the key exists.
- Aggregation and minimum group sizes can reduce disclosure, but do not prove anonymity.
- Apply access controls, retention limits, and jurisdiction-specific privacy requirements.